# Estadísticas descriptivas de los sitios candidatos

Cuadro descriptivo de la Sección 6.1 de la tesis, a partir de `sitios.csv` (`deduplicacion_sitios.ipynb`). El universo son los 299 sitios candidatos: sin cementerios y dentro de su municipio. El número de exhumaciones con hallazgo aproxima el número de personas recuperadas, porque en la base cada cuerpo exhumado suele ser un registro distinto. La base no registra costo, área prospectada ni duración de la misión.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

PATH_PROCESSED = Path("../data/Processed/")
MIN_SITIOS_MUNICIPIO = 5
DEPARTAMENTOS = {'05': 'Antioquia', '17': 'Caldas', '18': 'Caquetá', '20': 'Cesar', '47': 'Magdalena',
                 '50': 'Meta', '54': 'Norte de Santander', '66': 'Risaralda', '70': 'Sucre'}

sitios = pd.read_csv(PATH_PROCESSED / "sitios.csv", dtype={'municipio': str})
cand = sitios[(sitios['tipo_sitio'] == 'otro') & (~sitios['fuera_municipio'])].copy()
cand['departamento'] = cand['municipio'].str[:2].map(DEPARTAMENTOS)
print(f"Sitios candidatos: {len(cand)} en {cand['municipio'].nunique()} municipios")


In [ ]:
# ============================================================
# PANEL A: POR DEPARTAMENTO
# ============================================================
def panel(g):
    return pd.Series({
        'municipios': g['municipio'].nunique(),
        'sitios': len(g),
        'con_hallazgo': int((g['D'] == 1).sum()),
        'sin_hallazgo': int((g['D'] == 0).sum()),
        'sin_resultado': int(g['D'].isna().sum()),
        'tasa_hallazgo': g['D'].mean(),
        'exhumaciones_hallazgo': int(g['n_exhumaciones_hallazgo'].sum()),
    })

panel_a = cand.groupby('departamento').apply(panel).sort_values('sitios', ascending=False)
panel_a.loc['Total'] = panel(cand)
print(panel_a.round(2).to_string())


In [ ]:
# ============================================================
# PANEL B: ATRIBUTOS POR SITIO Y TASA DE HALLAZGO POR MUNICIPIO
# ============================================================
con_hallazgo = cand[cand['D'] == 1]
con_resultado = cand[cand['D'].notna()]
tasa_mun = con_resultado.groupby('municipio')['D'].agg(['size', 'mean'])
tasa_mun_grandes = tasa_mun[tasa_mun['size'] >= MIN_SITIOS_MUNICIPIO]['mean']

def fila(nombre, x):
    x = pd.Series(x).dropna()
    return {'variable': nombre, 'n': len(x), 'media': x.mean(), 'mediana': x.median(),
            'min': x.min(), 'max': x.max()}

panel_b = pd.DataFrame([
    fila('Registros por sitio', cand['n_registros']),
    fila('Diligencias distintas (visitas) por sitio', cand.loc[cand['n_visitas'] > 0, 'n_visitas']),
    fila('Exhumaciones con hallazgo por sitio con hallazgo', con_hallazgo['n_exhumaciones_hallazgo']),
    fila('Año de la primera diligencia', cand['anio_primera']),
    fila(f'Tasa de hallazgo por municipio (municipios con al menos {MIN_SITIOS_MUNICIPIO} sitios)',
         tasa_mun_grandes),
])
print(panel_b.round(2).to_string(index=False))

print(f"\nTasa de hallazgo por municipio, todos los municipios con resultado (n={len(tasa_mun)}):")
print(f"  municipios con tasa 1: {(tasa_mun['mean'] == 1).sum()}  |  con tasa 0: {(tasa_mun['mean'] == 0).sum()}")
print(f"  municipios con un solo sitio con resultado: {(tasa_mun['size'] == 1).sum()}")
print(f"\nTipo de disposición de los sitios candidatos:")
print(cand['disposicion'].fillna('Sin dato').value_counts().to_string())
print(f"\nSitios sin año de diligencia: {cand['anio_primera'].isna().sum()}")


In [ ]:
# ============================================================
# GUARDAR
# ============================================================
panel_a.to_csv(PATH_PROCESSED / "descriptivas_sitios_departamento.csv")
panel_b.to_csv(PATH_PROCESSED / "descriptivas_sitios_atributos.csv", index=False)
tasa_mun.to_csv(PATH_PROCESSED / "tasa_hallazgo_municipio.csv")
print("✓ Guardado")
